# ROS10 · Integration - a PID joint controller node, simulated joint, launch

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BrandonZabawa/robotics-daily-labs/blob/main/labs/03_ros2/ROS10_integration_a_pid_joint_controller_node_simulate.ipynb)

**Track:** ROS 2 · **Skill axis (sheet):** Control and motion · **Time:** about 75 min

**Skill Hub concepts:** ROS 2 nodes and topics; ROS 2 launch files; ROS 2 quality of service

**Level 1** (the base exercises). Harder levels go at the end of this notebook.

**Languages:** each node task has an optional C++ (rclcpp) cell after it. The grader builds it with colcon and runs the same checker against it.

> Cold attempt rules: notes closed, no AI, no internet until you run `lab.score()`. Fill every `TODO`, run the check under it, then do the checkpoint.

In [ ]:
#@title ⚙️ Setup: run this cell first
import os, sys, pathlib, subprocess
REPO_URL = "https://github.com/BrandonZabawa/robotics-daily-labs"   # or your fork
BRANCH = "main"                                  # the branch that holds your added levels
_root = os.environ.get("RDL_ROOT")
_root = pathlib.Path(_root) if _root else next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "roboprep").is_dir()), None)
if _root is None:
    _root = pathlib.Path("/content/robotics-daily-labs")
    if _root.exists():
        subprocess.run(["git", "-C", str(_root), "fetch", "-q", "origin", BRANCH])
        subprocess.run(["git", "-C", str(_root), "checkout", "-q", BRANCH])
        subprocess.run(["git", "-C", str(_root), "pull", "-q", "origin", BRANCH])
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, str(_root)], check=True)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from roboprep import Lab
lab = Lab("ROS10")

## Map: a complete little control system

```
 /joint/setpoint (Float64, RELIABLE + TRANSIENT_LOCAL) ──> [joint_pid] ── 100 Hz timer ──> /joint/effort (Float64, volts)
                                                             ^                                  │
                                      /joint_states (JointState, 200 Hz) <── [joint_sim]  <─────┘
                                                                              ω̇ = 10u - ω,  θ̇ = ω,  |u| <= 12 V
```

Plant: $\dfrac{\theta(s)}{u(s)} = \dfrac{10}{s(s+1)}$. With PD on measurement $u = k_p e - k_d\,\omega$ the closed loop is
$s^2 + (1 + 10k_d)s + 10k_p = 0$, so $\omega_n = \sqrt{10 k_p}$ and $\zeta = (1 + 10 k_d)/(2\omega_n)$.
A small $k_i$ removes steady-state error from friction or load.

| Design choice | Why |
|---|---|
| Timer-driven control at a fixed rate | constant $\Delta t$ for the integral, independent of message arrival |
| Derivative from the measured velocity | no derivative kick on setpoint steps, no numerical differentiation noise |
| Latched setpoint QoS | a controller restarted mid-run picks up the last setpoint |
| Gains as parameters | tune from the launch file or live with `ros2 param set` |

In [ ]:
from roboprep import ros

In [ ]:
%%writefile joint_sim.py
import rclpy
from rclpy.executors import ExternalShutdownException
from rclpy.node import Node
from sensor_msgs.msg import JointState
from std_msgs.msg import Float64


class JointSim(Node):
    def __init__(self):
        super().__init__("joint_sim")
        self.u, self.w, self.th = 0.0, 0.0, 0.0
        self.create_subscription(Float64, "/joint/effort", lambda m: setattr(self, "u", max(-12.0, min(12.0, m.data))), 10)
        self.pub = self.create_publisher(JointState, "/joint_states", 10)
        self.create_timer(0.005, self.tick)

    def tick(self):
        for _ in range(5):                      # 1 kHz integration inside a 200 Hz publish
            self.w += (10.0 * self.u - self.w) * 0.001
            self.th += self.w * 0.001
        js = JointState()
        js.header.stamp = self.get_clock().now().to_msg()
        js.name, js.position, js.velocity = ["joint1"], [self.th], [self.w]
        self.pub.publish(js)


def main():
    rclpy.init()
    try:
        rclpy.spin(JointSim())
    except (KeyboardInterrupt, ExternalShutdownException):
        pass


if __name__ == "__main__":
    main()

## Task 1: the controller node
Node `joint_pid`; parameters `kp` (3.6), `ki` (0.3), `kd` (0.86); output limit 12 V with conditional integration
(lab M03); latched setpoint subscription; publish effort from a 100 Hz timer (publish 0 until both a setpoint and a
joint state have arrived).

In [ ]:
%%writefile pid_node.py
import rclpy
from rclpy.executors import ExternalShutdownException
from rclpy.node import Node
from rclpy.qos import DurabilityPolicy, QoSProfile, ReliabilityPolicy
from sensor_msgs.msg import JointState
from std_msgs.msg import Float64


class JointPid(Node):
    def __init__(self):
        super().__init__("joint_pid")
        self.kp = self.declare_parameter("kp", 3.6).value
        self.ki = self.declare_parameter("ki", 0.3).value
        self.kd = self.declare_parameter("kd", 0.86).value
        self.dt, self.u_max = 0.01, 12.0
        self.sp = self.pos = self.vel = None
        self.integ = 0.0
        latched = QoSProfile(depth=1, reliability=ReliabilityPolicy.RELIABLE, durability=DurabilityPolicy.TRANSIENT_LOCAL)
        self.pub = self.create_publisher(Float64, "/joint/effort", 10)
        # TODO: two subscriptions (setpoint with the latched QoS) and a 100 Hz timer calling self.control

    def on_sp(self, msg):
        self.sp = msg.data

    def on_js(self, msg):
        if "joint1" in msg.name:
            k = msg.name.index("joint1")
            self.pos, self.vel = msg.position[k], msg.velocity[k]

    def control(self):
        self.pub.publish(Float64(data=0.0))   # TODO: PID with anti-windup, derivative from self.vel


def main():
    rclpy.init()
    try:
        rclpy.spin(JointPid())
    except (KeyboardInterrupt, ExternalShutdownException):
        pass


if __name__ == "__main__":
    main()

## Task 2: the launch file
Start `joint_sim.py` and `pid_node.py`, passing `kp:=3.6 ki:=0.3 kd:=0.86` explicitly as parameters.

In [ ]:
%%writefile control.launch.py
from launch import LaunchDescription
from launch.actions import ExecuteProcess


def generate_launch_description():
    return LaunchDescription([
        # TODO
    ])

Run a step and plot what the robot did.

In [ ]:
%%writefile step_logger.py
import json, time, rclpy
from rclpy.node import Node
from rclpy.qos import DurabilityPolicy, QoSProfile, ReliabilityPolicy
from sensor_msgs.msg import JointState
from std_msgs.msg import Float64
rclpy.init()
n = Node("step_logger")
q = QoSProfile(depth=1, reliability=ReliabilityPolicy.RELIABLE, durability=DurabilityPolicy.TRANSIENT_LOCAL)
sp = n.create_publisher(Float64, "/joint/setpoint", q)
log, t0 = [], time.time()
n.create_subscription(JointState, "/joint_states", lambda m: log.append((time.time() - t0, m.position[0])), 10)
while n.count_subscribers("/joint/setpoint") == 0 and time.time() - t0 < 15:
    rclpy.spin_once(n, timeout_sec=0.1)
log.clear(); t0 = time.time()
sp.publish(Float64(data=1.0))
while time.time() - t0 < 4.0:
    rclpy.spin_once(n, timeout_sec=0.01)
sp.publish(Float64(data=-0.5))
while time.time() - t0 < 8.0:
    rclpy.spin_once(n, timeout_sec=0.01)
print(json.dumps(log))

In [ ]:
import json, time
import matplotlib.pyplot as plt
ros.start("ros2 launch control.launch.py", log="launch.log")
time.sleep(3)
out = ros.ros("python step_logger.py", timeout=60, show=False).strip().splitlines()
ros.stop_all()
try:
    data = json.loads(out[-1])
    t, th = zip(*data)
    plt.plot(t, th); plt.axhline(1.0, ls="--", c="k"); plt.axhline(-0.5, ls="--", c="k")
    plt.xlabel("s"); plt.ylabel("joint1 position (rad)"); plt.show()
except Exception as e:
    print("no data yet:", out[-3:] if out else e)

In [ ]:
lab.check("closed_loop")

In [ ]:
%%writefile pid_node.cpp
// C++ (rclcpp) version of Task 2 (optional, same checker). Grade it with lab.check("closed_loop", "pid_node.cpp"):
// the grader builds it with colcon as package rdl_pid_node, executable pid_node (depends on rclcpp, std_msgs, sensor_msgs).
#include <algorithm>
#include <chrono>
#include <optional>
#include <rclcpp/rclcpp.hpp>
#include <sensor_msgs/msg/joint_state.hpp>
#include <std_msgs/msg/float64.hpp>
using namespace std::chrono_literals;

class JointPid : public rclcpp::Node {
public:
    JointPid() : Node("joint_pid") {
        kp_ = declare_parameter("kp", 3.6);
        ki_ = declare_parameter("ki", 0.3);
        kd_ = declare_parameter("kd", 0.86);
        pub_ = create_publisher<std_msgs::msg::Float64>("/joint/effort", 10);
        // TODO: /joint/setpoint subscription (latched QoS: depth 1, reliable, transient local),
        //       /joint_states subscription (read joint1 position and velocity), 100 Hz timer calling control()
    }
private:
    void control() {
        std_msgs::msg::Float64 out;
        out.data = 0.0;   // TODO: PID with anti-windup, derivative from the measured velocity, clamp to 12
        pub_->publish(out);
    }
    double kp_, ki_, kd_, dt_ = 0.01, u_max_ = 12.0, integ_ = 0.0, vel_ = 0.0;
    std::optional<double> sp_, pos_;
    rclcpp::Publisher<std_msgs::msg::Float64>::SharedPtr pub_;
};

int main(int argc, char **argv) {
    rclcpp::init(argc, argv);
    rclcpp::spin(std::make_shared<JointPid>());
    rclcpp::shutdown();
}

In [ ]:
# lab.check("closed_loop", "pid_node.cpp")  # uncomment after writing the C++ cell above

## Checkpoint: Math
**M1 (Apply).** Control period in seconds at 100 Hz?

**M2 (Apply).** Closed-loop natural frequency $\omega_n$ (rad/s) with $k_p = 3.6$?

In [ ]:
lab.answer("M1", None)
lab.answer("M2", None)

## Checkpoint: Concept
**K1 (Apply).** The PID node crashes and is restarted by the launch system 2 s later. Which setpoint QoS lets it
resume the last commanded position without anyone republishing?
- a) BEST_EFFORT, VOLATILE
- b) RELIABLE, TRANSIENT_LOCAL, depth 1
- c) RELIABLE, VOLATILE, depth 100

**K2 (Evaluate).** This node runs control in a Python timer on a normal Linux kernel. List what you would change to
make the loop deterministic on the real arm.

In [ ]:
lab.choose("K1", None)

In [ ]:
lab.explain("K2", """
Your explanation here.
""")

In [ ]:
lab.self_score({"K2": 0})

## Level 2 and beyond: harder exercises you add

Add each new exercise above the score cell, in this order:

1. A markdown cell: what to build and why (a table or diagram if it helps).
2. A code cell with your stub (Python, or a `%%writefile` C/C++ cell that includes `ros10.h`).
3. In `roboprep/checks/ros10.py`, a grader: `def check_L2_<name>(fn): assert ...` (for C/C++ also
   `def c_L2_<name>(lib): return (callable, ...)` built with `roboprep.cbind.Functions`). Every `check_` function in that
   file counts toward the code score automatically.
4. `lab.check("L2_<name>", your_function)` (or the C/C++ file name) under your stub.

Commit the notebook and the grader to your branch and set `BRANCH` in the setup cell so Colab clones it.

In [ ]:
# Level 2: your exercises go here (see the cell above)

In [ ]:
lab.score()